<a href="https://colab.research.google.com/github/ProfessorDong/Deep-Learning-Course-Examples/blob/master/CNN_Examples/CNN_Demo5_YOLO_Object_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Object Detection with YOLO: From Theory to Practice

**ELC 5365 - Deep Learning, Baylor University**

In this notebook, we will explore **object detection** -- one of the most important tasks in computer vision. We will:

1. Understand the difference between classification and detection
2. Implement **Intersection over Union (IoU)** from scratch
3. Implement **Non-Maximum Suppression (NMS)** step by step
4. Visualize how **YOLO** divides an image into a grid
5. Run a **pretrained object detector** on real images
6. Detect multiple objects in a single image
7. Visualize **anchor boxes** used in modern detectors
8. Compare detection approaches (R-CNN family vs. YOLO/SSD)

---

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.colors as mcolors
import numpy as np
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print(f"Torchvision version: {torchvision.__version__}")

---
## Section 1: Classification vs. Detection

In computer vision, there is a hierarchy of tasks with increasing complexity:

| Task | What It Does | Example Output |
|------|-------------|----------------|
| **Classification** | "This is a cat" | Single label |
| **Classification + Localization** | "This is a cat, located here" | Single label + one bounding box |
| **Object Detection** | "There are 2 dogs and 1 cat, located here, here, and here" | Multiple labels + multiple bounding boxes |
| **Instance Segmentation** | Pixel-level masks for each object | Multiple labels + pixel masks |

### The Evolution of Object Detection

The field has evolved dramatically over the past decade:

1. **R-CNN (2014)** -- Region proposals + CNN classification (very slow: ~47s per image)
2. **Fast R-CNN (2015)** -- Share CNN computation across regions (faster: ~2 FPS)
3. **Faster R-CNN (2015)** -- Replace selective search with a **Region Proposal Network** (~7 FPS)
4. **YOLO (2016)** -- **"You Only Look Once"** -- single-shot detection in one pass (~45 FPS)
5. **SSD (2016)** -- Single Shot MultiBox Detector -- multi-scale single-shot detection (~59 FPS)

**Key insight:** The trend moved from **multi-stage** pipelines (propose regions, then classify) to **single-stage** detectors (predict boxes and classes in one forward pass), dramatically improving speed while maintaining competitive accuracy.

In [ ]:
# Visual illustration of Classification vs Detection
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Create a simple synthetic scene with colored rectangles representing objects
scene = np.ones((200, 300, 3)) * 0.9  # Light gray background
# "Cat" - orange rectangle
scene[60:140, 30:110, :] = [1.0, 0.6, 0.2]
# "Dog" - brown rectangle
scene[80:170, 170:270, :] = [0.6, 0.4, 0.2]

# 1) Classification
axes[0].imshow(scene)
axes[0].set_title('Classification\n"This image contains: cat, dog"', fontsize=13, fontweight='bold')
axes[0].axis('off')

# 2) Classification + Localization (single object)
axes[1].imshow(scene)
rect = patches.Rectangle((30, 60), 80, 80, linewidth=3, edgecolor='lime', facecolor='none')
axes[1].add_patch(rect)
axes[1].text(32, 55, 'cat (0.95)', color='lime', fontsize=12, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
axes[1].set_title('Classification + Localization\n"Cat is here" (single object)', fontsize=13, fontweight='bold')
axes[1].axis('off')

# 3) Object Detection (multiple objects)
axes[2].imshow(scene)
rect1 = patches.Rectangle((30, 60), 80, 80, linewidth=3, edgecolor='lime', facecolor='none')
rect2 = patches.Rectangle((170, 80), 100, 90, linewidth=3, edgecolor='cyan', facecolor='none')
axes[2].add_patch(rect1)
axes[2].add_patch(rect2)
axes[2].text(32, 55, 'cat (0.95)', color='lime', fontsize=12, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
axes[2].text(172, 75, 'dog (0.89)', color='cyan', fontsize=12, fontweight='bold',
             bbox=dict(boxstyle='round,pad=0.2', facecolor='black', alpha=0.7))
axes[2].set_title('Object Detection\n"Cat here, dog here" (multiple objects)', fontsize=13, fontweight='bold')
axes[2].axis('off')

plt.tight_layout()
plt.show()

---
## Section 2: Intersection over Union (IoU) -- Implemented from Scratch

**Intersection over Union (IoU)** is the most fundamental metric in object detection. It measures how well a predicted bounding box overlaps with the ground truth.

$$\text{IoU} = \frac{\text{Area of Intersection}}{\text{Area of Union}} = \frac{|A \cap B|}{|A \cup B|}$$

- **IoU = 1.0**: Perfect overlap (predicted box = ground truth box)
- **IoU = 0.0**: No overlap at all
- **IoU >= 0.5**: Typically considered a **"correct" detection** (this is the standard threshold used in PASCAL VOC)
- **IoU >= 0.75**: A "strict" correct detection (used in COCO evaluation)

In [ ]:
def compute_iou(box1, box2):
    """
    Compute IoU between two bounding boxes.
    Each box is [x1, y1, x2, y2] where (x1,y1) is top-left and (x2,y2) is bottom-right.
    """
    # Compute intersection coordinates
    x1_inter = max(box1[0], box2[0])
    y1_inter = max(box1[1], box2[1])
    x2_inter = min(box1[2], box2[2])
    y2_inter = min(box1[3], box2[3])
    
    # Compute intersection area (clamp to 0 if no overlap)
    inter_width = max(0, x2_inter - x1_inter)
    inter_height = max(0, y2_inter - y1_inter)
    intersection = inter_width * inter_height
    
    # Compute areas of each box
    area1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    area2 = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    # Compute union
    union = area1 + area2 - intersection
    
    # Compute IoU
    iou = intersection / union if union > 0 else 0
    
    return iou, intersection, union

# Quick test
box_a = [50, 50, 200, 200]   # 150 x 150 box
box_b = [100, 100, 250, 250] # 150 x 150 box
iou, inter, union = compute_iou(box_a, box_b)
print(f"Box A: {box_a}")
print(f"Box B: {box_b}")
print(f"Intersection area: {inter}")
print(f"Union area: {union}")
print(f"IoU: {iou:.4f}")

In [ ]:
def visualize_iou(box1, box2, ax, title=""):
    """
    Visualize two bounding boxes, their intersection, and IoU score.
    """
    iou, intersection, union = compute_iou(box1, box2)
    
    # Draw background
    ax.set_xlim(0, 350)
    ax.set_ylim(0, 350)
    ax.set_aspect('equal')
    ax.invert_yaxis()
    ax.set_facecolor('#f0f0f0')
    
    # Draw box 1 (Ground Truth) - filled with semi-transparent blue
    rect1 = patches.Rectangle(
        (box1[0], box1[1]), box1[2]-box1[0], box1[3]-box1[1],
        linewidth=3, edgecolor='#2196F3', facecolor='#2196F3', alpha=0.25, label='Ground Truth'
    )
    ax.add_patch(rect1)
    
    # Draw box 2 (Prediction) - filled with semi-transparent red
    rect2 = patches.Rectangle(
        (box2[0], box2[1]), box2[2]-box2[0], box2[3]-box2[1],
        linewidth=3, edgecolor='#F44336', facecolor='#F44336', alpha=0.25, label='Prediction'
    )
    ax.add_patch(rect2)
    
    # Draw intersection area - filled with green
    x1_inter = max(box1[0], box2[0])
    y1_inter = max(box1[1], box2[1])
    x2_inter = min(box1[2], box2[2])
    y2_inter = min(box1[3], box2[3])
    
    if x2_inter > x1_inter and y2_inter > y1_inter:
        rect_inter = patches.Rectangle(
            (x1_inter, y1_inter), x2_inter-x1_inter, y2_inter-y1_inter,
            linewidth=2, edgecolor='#4CAF50', facecolor='#4CAF50', alpha=0.5, label='Intersection'
        )
        ax.add_patch(rect_inter)
    
    # Draw outlines on top
    rect1_outline = patches.Rectangle(
        (box1[0], box1[1]), box1[2]-box1[0], box1[3]-box1[1],
        linewidth=3, edgecolor='#2196F3', facecolor='none'
    )
    rect2_outline = patches.Rectangle(
        (box2[0], box2[1]), box2[2]-box2[0], box2[3]-box2[1],
        linewidth=3, edgecolor='#F44336', facecolor='none', linestyle='--'
    )
    ax.add_patch(rect1_outline)
    ax.add_patch(rect2_outline)
    
    # IoU score display
    color = '#4CAF50' if iou >= 0.5 else ('#FF9800' if iou >= 0.3 else '#F44336')
    verdict = 'CORRECT' if iou >= 0.5 else 'INCORRECT'
    ax.set_title(f"{title}\nIoU = {iou:.3f} ({verdict})", fontsize=14, fontweight='bold', color=color)
    ax.legend(loc='lower right', fontsize=9)


# Demonstrate three cases: high, medium, and low IoU
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Case 1: High overlap (IoU ~ 0.8)
gt_box = [80, 80, 250, 250]
pred_high = [90, 85, 260, 255]
visualize_iou(gt_box, pred_high, axes[0], title="High Overlap")

# Case 2: Medium overlap (IoU ~ 0.4)
pred_medium = [150, 140, 320, 310]
visualize_iou(gt_box, pred_medium, axes[1], title="Medium Overlap")

# Case 3: Low overlap (IoU ~ 0.1)
pred_low = [210, 210, 340, 340]
visualize_iou(gt_box, pred_low, axes[2], title="Low Overlap")

plt.suptitle('IoU (Intersection over Union) -- The Detection Quality Metric',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("Blue solid box = Ground Truth | Red dashed box = Prediction | Green = Intersection")
print("\nIoU >= 0.5 is typically considered a 'correct' detection (PASCAL VOC standard).")

---
## Section 3: Non-Maximum Suppression (NMS) -- Step by Step

When a detector runs on an image, it typically produces **hundreds or thousands** of overlapping bounding boxes for each object. **Non-Maximum Suppression (NMS)** is the algorithm that filters these down to one box per object.

### NMS Algorithm:
1. **Sort** all boxes by their confidence score (highest first)
2. **Select** the box with the highest confidence -- add it to the final results
3. **Compute IoU** of this box with all remaining boxes
4. **Remove** any remaining box whose IoU with the selected box exceeds a threshold (e.g., 0.5)
5. **Repeat** from step 2 until no boxes remain

The intuition: if two boxes overlap significantly, they are probably detecting the **same object**, so we keep only the one with higher confidence.

In [ ]:
def nms(boxes, scores, iou_threshold=0.5):
    """
    Non-Maximum Suppression implemented from scratch.
    
    Args:
        boxes: list of [x1, y1, x2, y2]
        scores: list of confidence scores
        iou_threshold: IoU threshold for suppression
    
    Returns:
        keep_indices: indices of boxes to keep
        steps: list of (selected_idx, suppressed_indices) for visualization
    """
    # Sort indices by score (descending)
    indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    
    keep_indices = []
    steps = []  # Track each step for visualization
    
    while len(indices) > 0:
        # Pick the box with highest score
        current = indices[0]
        keep_indices.append(current)
        
        # Compare with all remaining boxes
        remaining = indices[1:]
        suppressed = []
        new_indices = []
        
        for idx in remaining:
            iou_val, _, _ = compute_iou(boxes[current], boxes[idx])
            if iou_val > iou_threshold:
                suppressed.append(idx)
            else:
                new_indices.append(idx)
        
        steps.append((current, suppressed))
        indices = new_indices
    
    return keep_indices, steps

print("NMS function defined successfully.")

In [ ]:
# Generate synthetic detection scenario:
# A single object (orange rectangle) with 10 overlapping predicted bounding boxes
np.random.seed(42)

# The "object" is centered around (200, 150) with size ~120x100
obj_center_x, obj_center_y = 200, 150
obj_w, obj_h = 120, 100

# Generate 10 overlapping boxes around the object (simulating raw detector output)
n_boxes = 10
boxes = []
scores = []

for i in range(n_boxes):
    # Random offset from the true object center
    dx = np.random.randint(-30, 30)
    dy = np.random.randint(-25, 25)
    dw = np.random.randint(-20, 25)
    dh = np.random.randint(-20, 25)
    
    x1 = obj_center_x - obj_w//2 + dx
    y1 = obj_center_y - obj_h//2 + dy
    x2 = x1 + obj_w + dw
    y2 = y1 + obj_h + dh
    
    boxes.append([x1, y1, x2, y2])
    # Boxes closer to the true center get higher scores
    dist = np.sqrt(dx**2 + dy**2)
    score = max(0.3, min(0.99, 0.95 - dist * 0.015 + np.random.uniform(-0.1, 0.1)))
    scores.append(round(score, 2))

print(f"Generated {n_boxes} bounding boxes (simulating raw detector output):")
for i, (box, score) in enumerate(zip(boxes, scores)):
    print(f"  Box {i}: {box}  score={score:.2f}")

In [ ]:
# Visualize Before NMS vs After NMS
keep_indices, nms_steps = nms(boxes, scores, iou_threshold=0.5)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Color map based on confidence
cmap = plt.cm.RdYlGn  # Red (low) -> Yellow (medium) -> Green (high)

for ax_idx, (ax, title, show_indices) in enumerate([
    (axes[0], 'BEFORE NMS (All 10 Detections)', list(range(n_boxes))),
    (axes[1], f'AFTER NMS ({len(keep_indices)} Detection(s) Remaining)', keep_indices)
]):
    # Draw the "scene" background
    scene = np.ones((300, 400, 3)) * 0.85
    # Draw the actual object
    obj_x1 = obj_center_x - obj_w//2
    obj_y1 = obj_center_y - obj_h//2
    scene[obj_y1:obj_y1+obj_h, obj_x1:obj_x1+obj_w] = [1.0, 0.65, 0.2]
    ax.imshow(scene)
    
    # Draw bounding boxes
    for i in show_indices:
        color = cmap(scores[i])
        rect = patches.Rectangle(
            (boxes[i][0], boxes[i][1]),
            boxes[i][2] - boxes[i][0],
            boxes[i][3] - boxes[i][1],
            linewidth=2.5, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        ax.text(boxes[i][0], boxes[i][1]-3, f'{scores[i]:.2f}',
                color='white', fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.15', facecolor=color, alpha=0.85))
    
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.axis('off')

plt.suptitle('Non-Maximum Suppression (NMS) -- Removing Redundant Detections',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("NMS removes redundant detections. Each object should be detected only once.")

In [ ]:
# Step-by-step NMS visualization
n_steps = len(nms_steps)
fig, axes = plt.subplots(1, n_steps + 1, figsize=(6 * (n_steps + 1), 5))
if n_steps + 1 == 1:
    axes = [axes]

# Track which boxes have been suppressed so far
all_suppressed = set()
all_kept = []

for step_idx, (selected, suppressed) in enumerate(nms_steps):
    ax = axes[step_idx]
    scene = np.ones((300, 400, 3)) * 0.85
    obj_x1 = obj_center_x - obj_w//2
    obj_y1 = obj_center_y - obj_h//2
    scene[obj_y1:obj_y1+obj_h, obj_x1:obj_x1+obj_w] = [1.0, 0.65, 0.2]
    ax.imshow(scene)
    
    all_kept.append(selected)
    
    # Draw all remaining boxes at this step
    for i in range(n_boxes):
        if i in all_suppressed:
            continue  # Already removed in previous steps
        
        if i == selected:
            # The selected box -- thick green
            color = '#4CAF50'
            lw = 4
            ls = '-'
        elif i in suppressed:
            # Being suppressed this step -- red dashed
            color = '#F44336'
            lw = 2.5
            ls = '--'
        elif i in all_kept:
            # Previously kept
            color = '#4CAF50'
            lw = 3
            ls = '-'
        else:
            # Still in the pool
            color = '#9E9E9E'
            lw = 1.5
            ls = '-'
        
        rect = patches.Rectangle(
            (boxes[i][0], boxes[i][1]),
            boxes[i][2] - boxes[i][0],
            boxes[i][3] - boxes[i][1],
            linewidth=lw, edgecolor=color, facecolor='none', linestyle=ls
        )
        ax.add_patch(rect)
        ax.text(boxes[i][0], boxes[i][1]-3, f'{scores[i]:.2f}',
                color='white', fontsize=8, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.1', facecolor=color, alpha=0.85))
    
    ax.set_title(f'Step {step_idx+1}: Select box {selected} (score={scores[selected]:.2f})\n'
                 f'Suppress {len(suppressed)} box(es) with IoU > 0.5',
                 fontsize=11, fontweight='bold')
    ax.axis('off')
    
    all_suppressed.update(suppressed)

# Final result
ax = axes[-1]
scene = np.ones((300, 400, 3)) * 0.85
scene[obj_y1:obj_y1+obj_h, obj_x1:obj_x1+obj_w] = [1.0, 0.65, 0.2]
ax.imshow(scene)
for i in keep_indices:
    rect = patches.Rectangle(
        (boxes[i][0], boxes[i][1]),
        boxes[i][2] - boxes[i][0],
        boxes[i][3] - boxes[i][1],
        linewidth=4, edgecolor='#4CAF50', facecolor='none'
    )
    ax.add_patch(rect)
    ax.text(boxes[i][0], boxes[i][1]-3, f'{scores[i]:.2f}',
            color='white', fontsize=10, fontweight='bold',
            bbox=dict(boxstyle='round,pad=0.15', facecolor='#4CAF50', alpha=0.9))
ax.set_title(f'Final Result\n{len(keep_indices)} detection(s) kept', fontsize=11, fontweight='bold')
ax.axis('off')

plt.suptitle('NMS Step by Step: Green = Kept, Red Dashed = Suppressed, Gray = Remaining',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Section 4: How YOLO Works -- The Grid

**YOLO (You Only Look Once)** takes a fundamentally different approach from R-CNN-based methods:

### The YOLO Pipeline:
1. **Divide** the image into an **S x S grid** (S=7 in YOLOv1)
2. Each grid cell predicts **B bounding boxes** (B=2 in YOLOv1), each with:
   - 4 coordinates: (x, y, w, h) relative to the grid cell
   - 1 confidence score: P(Object) x IoU(pred, truth)
3. Each grid cell also predicts **C class probabilities** (C=20 for PASCAL VOC)
4. The output is a single tensor of shape: **S x S x (B*5 + C)** = 7 x 7 x 30

### Key Rule:
The grid cell containing the **center** of an object is **responsible** for detecting that object.

This is why YOLO is so fast -- it makes all predictions in a **single forward pass** through the network!

In [ ]:
# Load a CIFAR-10 image and overlay the YOLO grid
transform_simple = transforms.Compose([transforms.ToTensor()])
cifar_test = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_simple
)
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# Find one image per class
sample_indices = {}
for idx in range(len(cifar_test)):
    _, label = cifar_test[idx]
    if label not in sample_indices:
        sample_indices[label] = idx
    if len(sample_indices) == 10:
        break

# Pick a horse image for demonstration
img_tensor, label = cifar_test[sample_indices[7]]  # 7 = horse
img_np = img_tensor.permute(1, 2, 0).numpy()

# Upscale for better visualization (YOLOv1 uses 448x448 input)
img_pil = Image.fromarray((img_np * 255).astype(np.uint8))
img_upscaled = img_pil.resize((448, 448), Image.NEAREST)
img_up_np = np.array(img_upscaled) / 255.0

print(f"Selected image class: {class_names[label]}")
print(f"Original size: {img_np.shape[:2]}, Upscaled to: {img_up_np.shape[:2]}")
print(f"YOLO divides this into a 7x7 grid (49 cells total).")

In [ ]:
# Visualize the YOLO 7x7 grid on the image
S = 7  # Grid size
img_size = 448
cell_size = img_size / S

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Panel 1: Original image with grid overlay
axes[0].imshow(img_up_np)
for i in range(S + 1):
    axes[0].axhline(y=i * cell_size, color='white', linewidth=1, alpha=0.7)
    axes[0].axvline(x=i * cell_size, color='white', linewidth=1, alpha=0.7)
axes[0].set_title(f'Step 1: Divide into {S}x{S} Grid', fontsize=14, fontweight='bold')
axes[0].axis('off')

# Panel 2: Highlight the "responsible" cell (containing the object center)
responsible_cells = [(3, 3)]  # (row, col) - center of the image

axes[1].imshow(img_up_np)
for i in range(S + 1):
    axes[1].axhline(y=i * cell_size, color='white', linewidth=1, alpha=0.5)
    axes[1].axvline(x=i * cell_size, color='white', linewidth=1, alpha=0.5)

for row, col in responsible_cells:
    rect = patches.Rectangle(
        (col * cell_size, row * cell_size), cell_size, cell_size,
        linewidth=3, edgecolor='#FF5722', facecolor='#FF5722', alpha=0.35
    )
    axes[1].add_patch(rect)
    axes[1].plot(col * cell_size + cell_size/2, row * cell_size + cell_size/2,
                 'r*', markersize=20, markeredgecolor='white', markeredgewidth=1)

axes[1].set_title('Step 2: Find "Responsible" Cell\n(contains object center)',
                   fontsize=14, fontweight='bold')
axes[1].axis('off')

# Panel 3: Show bounding box predictions from that cell
axes[2].imshow(img_up_np)
for i in range(S + 1):
    axes[2].axhline(y=i * cell_size, color='white', linewidth=0.5, alpha=0.3)
    axes[2].axvline(x=i * cell_size, color='white', linewidth=0.5, alpha=0.3)

# Simulated bounding box predictions (B=2 boxes per cell in YOLOv1)
# Box 1 - tighter fit (higher confidence)
bbox1 = patches.Rectangle((80, 50), 310, 370, linewidth=3,
                            edgecolor='#4CAF50', facecolor='none', linestyle='-')
# Box 2 - looser fit (lower confidence)
bbox2 = patches.Rectangle((50, 30), 360, 400, linewidth=2.5,
                            edgecolor='#FFC107', facecolor='none', linestyle='--')

axes[2].add_patch(bbox2)
axes[2].add_patch(bbox1)
axes[2].text(82, 42, 'Box 1: conf=0.85', color='white', fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='#4CAF50', alpha=0.85))
axes[2].text(52, 22, 'Box 2: conf=0.42', color='white', fontsize=10, fontweight='bold',
             bbox=dict(boxstyle='round', facecolor='#FFC107', alpha=0.85))

# Highlight responsible cell
for row, col in responsible_cells:
    rect = patches.Rectangle(
        (col * cell_size, row * cell_size), cell_size, cell_size,
        linewidth=2, edgecolor='#FF5722', facecolor='#FF5722', alpha=0.2
    )
    axes[2].add_patch(rect)

axes[2].set_title('Step 3: Each Cell Predicts B=2 Boxes\n(with confidence scores)',
                   fontsize=14, fontweight='bold')
axes[2].axis('off')

plt.suptitle(f'How YOLO Works: Grid-Based Detection (Image class: {class_names[label]})',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nYOLOv1 output tensor shape: {S} x {S} x (B*5 + C) = {S} x {S} x {2*5 + 20} = {S} x {S} x 30")
print(f"Each cell predicts: 2 bounding boxes (each with x, y, w, h, confidence) + 20 class probabilities")

---
## Section 5: Object Detection with Pretrained Models

Now let's use a **pretrained object detector** to perform real detection! We provide two options:

### Option A: Torchvision's Faster R-CNN (recommended for classroom -- more reliable)
- `torchvision.models.detection.fasterrcnn_resnet50_fpn` with pretrained COCO weights
- Detects 91 COCO object categories
- Very accurate and well-tested

### Option B: Ultralytics YOLOv5 via torch.hub
- `torch.hub.load('ultralytics/yolov5', 'yolov5s')` 
- Faster and more modern, but requires internet and may have dependency issues

We will use **Option A** as the primary approach for reliability in classroom settings.

In [ ]:
# ============================================================
# Option A: Torchvision Faster R-CNN (Primary -- Recommended)
# ============================================================
print("Loading Faster R-CNN with pretrained COCO weights...")
weights = FasterRCNN_ResNet50_FPN_Weights.DEFAULT
model = fasterrcnn_resnet50_fpn(weights=weights)
model.eval()
model = model.to(device)

# Get the COCO class names from the weights metadata
coco_classes = weights.meta["categories"]
print(f"Model loaded! Can detect {len(coco_classes)} COCO categories.")
print(f"Example categories: {coco_classes[:15]}...")

In [ ]:
# ============================================================
# Option B: YOLOv5 via torch.hub (Alternative -- uncomment to try)
# ============================================================
# Uncomment the lines below to try YOLOv5 instead:
#
# print("Loading YOLOv5 via torch.hub...")
# yolo_model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
# yolo_model.eval()
# print("YOLOv5 loaded!")
#
# To run YOLOv5 on an image:
# results = yolo_model(img_pil)  # Pass a PIL image directly
# results.show()                 # Display results
# results.print()                # Print detected objects

print("(YOLOv5 option is available above -- uncomment to use it.)")
print("We will proceed with Faster R-CNN for the rest of this demo.")

In [ ]:
# Helper function to run detection and draw results
def detect_and_draw(model, image_tensor, ax, title="", score_threshold=0.5,
                    class_names=None, max_detections=20):
    """
    Run Faster R-CNN on an image tensor and draw bounding boxes.
    
    Args:
        model: the detection model
        image_tensor: tensor of shape [3, H, W] with values in [0, 1]
        ax: matplotlib axis to draw on
        title: title for the plot
        score_threshold: minimum confidence to show a detection
        class_names: list of class names
        max_detections: maximum number of boxes to draw
    """
    # Run the model
    with torch.no_grad():
        predictions = model([image_tensor.to(device)])
    
    pred = predictions[0]
    det_boxes = pred['boxes'].cpu().numpy()
    det_labels = pred['labels'].cpu().numpy()
    det_scores = pred['scores'].cpu().numpy()
    
    # Display the image
    img_display = image_tensor.permute(1, 2, 0).cpu().numpy()
    ax.imshow(img_display)
    
    # Color palette for different classes
    color_list = ['#FF5722', '#2196F3', '#4CAF50', '#FFC107', '#9C27B0',
                  '#00BCD4', '#FF9800', '#E91E63', '#3F51B5', '#8BC34A',
                  '#795548', '#607D8B', '#CDDC39', '#F44336', '#009688']
    
    n_drawn = 0
    for i in range(len(det_boxes)):
        if det_scores[i] < score_threshold:
            continue
        if n_drawn >= max_detections:
            break
        
        x1, y1, x2, y2 = det_boxes[i]
        label_idx = det_labels[i]
        score = det_scores[i]
        
        color = color_list[label_idx % len(color_list)]
        
        # Draw bounding box
        rect = patches.Rectangle(
            (x1, y1), x2-x1, y2-y1,
            linewidth=2.5, edgecolor=color, facecolor='none'
        )
        ax.add_patch(rect)
        
        # Draw label
        label_text = f"{class_names[label_idx]}: {score:.2f}" if class_names else f"class {label_idx}: {score:.2f}"
        ax.text(x1, y1-4, label_text, color='white', fontsize=9, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', facecolor=color, alpha=0.85))
        n_drawn += 1
    
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.axis('off')
    
    return n_drawn

print("Detection and drawing helper function defined.")

---
## Section 6: Running Detection on Sample Images

Let's run our pretrained Faster R-CNN detector on several CIFAR-10 test images. Since CIFAR-10 images are tiny (32x32), we will **resize them** to a larger resolution that the detector can work with.

Note: CIFAR-10 images contain single, centered objects, so detections may be limited. The detector was trained on COCO, which has different categories -- but many overlap (airplane, car, cat, dog, bird, horse, truck).

In [ ]:
# Select specific CIFAR-10 images from interesting classes
# CIFAR-10 classes: airplane(0), automobile(1), bird(2), cat(3), deer(4),
#                   dog(5), frog(6), horse(7), ship(8), truck(9)
target_classes = [0, 1, 2, 3, 5, 7, 8, 9]  # airplane, auto, bird, cat, dog, horse, ship, truck
selected_images = []

# Get one image per target class
found_classes = set()
for idx in range(len(cifar_test)):
    img_t, lbl = cifar_test[idx]
    if lbl in target_classes and lbl not in found_classes:
        selected_images.append((img_t, lbl))
        found_classes.add(lbl)
    if len(found_classes) == len(target_classes):
        break

print(f"Selected {len(selected_images)} images for detection:")
for img_t, lbl in selected_images:
    print(f"  - {class_names[lbl]}")

In [ ]:
# Run detection on each selected CIFAR-10 image
n_images = len(selected_images)
cols = 4
rows = (n_images + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows))
axes = axes.flatten()

for i, (img_tensor_sel, label_sel) in enumerate(selected_images):
    # Resize to 224x224 for the detector (bilinear interpolation for smoother result)
    img_pil_sel = transforms.ToPILImage()(img_tensor_sel)
    img_resized_sel = img_pil_sel.resize((224, 224), Image.BILINEAR)
    img_tensor_resized = transforms.ToTensor()(img_resized_sel)
    
    n_det = detect_and_draw(
        model, img_tensor_resized, axes[i],
        title=f'CIFAR-10: {class_names[label_sel]}',
        score_threshold=0.3,
        class_names=coco_classes
    )
    axes[i].text(5, 215, f'{n_det} detection(s)', color='white', fontsize=9,
                 bbox=dict(boxstyle='round', facecolor='black', alpha=0.6))

# Hide unused axes
for j in range(i + 1, len(axes)):
    axes[j].axis('off')

plt.suptitle('Faster R-CNN Detection on CIFAR-10 Images (resized to 224x224)',
             fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\nNote: CIFAR-10 images are only 32x32 -- very low resolution for a detector.")
print("The detector still finds objects, but confidence may be lower than on high-res images.")

---
## Section 7: Detecting Multiple Objects

The true power of object detection shines when there are **multiple objects** in a scene. Let's create a **collage** by stitching together several CIFAR-10 images onto a larger canvas. This simulates a real-world scene with multiple objects at different locations.

This demonstrates the key advantage of **detection** over **classification**: we can find and localize **every object** in the image, not just assign a single label.

In [ ]:
# Create a collage image by placing multiple CIFAR-10 images on a larger canvas
canvas_size = 448
canvas = np.ones((canvas_size, canvas_size, 3), dtype=np.float32) * 0.3  # Dark gray background

# Add some texture to the background
np.random.seed(123)
noise = np.random.uniform(-0.05, 0.05, canvas.shape).astype(np.float32)
canvas = np.clip(canvas + noise, 0, 1)

# Select 6 images and place them at different positions on the canvas
# Each CIFAR-10 image is 32x32; we upscale them to different sizes
placements = [
    # (class_idx, position_x, position_y, scale_factor)
    (0, 20, 20, 4),      # airplane - top left, 128x128
    (1, 280, 30, 3),     # automobile - top right, 96x96
    (3, 50, 250, 4),     # cat - bottom left, 128x128
    (5, 270, 200, 5),    # dog - bottom right, 160x160
    (2, 180, 130, 3),    # bird - center, 96x96
    (9, 160, 10, 3),     # truck - top center, 96x96
]

placed_objects = []
for cls_idx, px, py, scale in placements:
    img_t, _ = cifar_test[sample_indices[cls_idx]]
    img_pil_c = transforms.ToPILImage()(img_t)
    new_size = 32 * scale
    img_resized_c = img_pil_c.resize((new_size, new_size), Image.BILINEAR)
    img_arr = np.array(img_resized_c) / 255.0
    
    # Clip to canvas bounds
    end_x = min(px + new_size, canvas_size)
    end_y = min(py + new_size, canvas_size)
    paste_w = end_x - px
    paste_h = end_y - py
    
    canvas[py:end_y, px:end_x] = img_arr[:paste_h, :paste_w]
    placed_objects.append((class_names[cls_idx], px, py, end_x, end_y))

print("Collage created with the following objects:")
for name, x1, y1, x2, y2 in placed_objects:
    print(f"  - {name} at ({x1}, {y1}) to ({x2}, {y2})")

In [ ]:
# Run detection on the collage
canvas_tensor = torch.tensor(canvas).permute(2, 0, 1).float()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# Left: Original collage with ground truth labels
axes[0].imshow(canvas)
for name, x1, y1, x2, y2 in placed_objects:
    rect = patches.Rectangle(
        (x1, y1), x2-x1, y2-y1,
        linewidth=2, edgecolor='white', facecolor='none', linestyle='--'
    )
    axes[0].add_patch(rect)
    axes[0].text(x1+2, y1-5, name, color='white', fontsize=10, fontweight='bold',
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='gray', alpha=0.7))
axes[0].set_title('Collage Image (Ground Truth Placements)', fontsize=14, fontweight='bold')
axes[0].axis('off')

# Right: Detection results from Faster R-CNN
n_det = detect_and_draw(
    model, canvas_tensor, axes[1],
    title='Faster R-CNN Detections',
    score_threshold=0.3,
    class_names=coco_classes
)

plt.suptitle(f'Multi-Object Detection on Collage Image ({n_det} object(s) detected)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nThe detector found {n_det} object(s) in the collage.")
print("This demonstrates the key advantage of detection over classification:")
print("We can find and localize EVERY object, not just assign one label to the whole image.")

In [ ]:
# Print detailed detection results for the collage
with torch.no_grad():
    predictions = model([canvas_tensor.to(device)])

pred = predictions[0]
print("Detailed Detection Results:")
print("=" * 65)
print(f"{'#':<4} {'Class':<20} {'Score':<8} {'Bounding Box (x1,y1,x2,y2)'}")
print("-" * 65)

for i in range(len(pred['boxes'])):
    score = pred['scores'][i].item()
    if score < 0.3:
        continue
    label_idx = pred['labels'][i].item()
    box = pred['boxes'][i].cpu().numpy()
    print(f"{i+1:<4} {coco_classes[label_idx]:<20} {score:<8.3f} [{box[0]:.0f}, {box[1]:.0f}, {box[2]:.0f}, {box[3]:.0f}]")

print("=" * 65)

---
## Section 8: Anchor Boxes Visualization

Modern detectors like **Faster R-CNN** and **SSD** use **anchor boxes** (also called **prior boxes** or **default boxes**) as reference templates for detection.

### What are Anchor Boxes?
- **Fixed-shape reference boxes** placed at each spatial location in the feature map
- Come in multiple **scales** (small, medium, large) and **aspect ratios** (tall, square, wide)
- The network does **not** predict bounding boxes from scratch -- instead, it predicts **offsets** (adjustments) from these anchors
- This makes learning much easier: the network only needs to learn small refinements

### In Faster R-CNN's Region Proposal Network (RPN):
- **3 scales** x **3 aspect ratios** = **9 anchors** per spatial location
- On a typical feature map, this generates approximately **20,000 anchors** per image
- NMS and confidence scoring reduce these to the final ~300 region proposals

In [ ]:
# Visualize anchor boxes
from matplotlib.lines import Line2D

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# Use the upscaled CIFAR-10 horse image as background
img_bg = img_up_np.copy()

# Define anchor box parameters (Faster R-CNN style)
anchor_scales = [64, 128, 256]       # 3 scales
aspect_ratios = [0.5, 1.0, 2.0]      # 3 aspect ratios (tall, square, wide)
center_x, center_y = 224, 224        # Center of the image

scale_colors = ['#FF5722', '#4CAF50', '#2196F3']  # Red, Green, Blue for each scale

# Panel 1: Show 9 anchors at a single point, colored by scale
axes[0].imshow(img_bg)
axes[0].plot(center_x, center_y, 'r+', markersize=20, markeredgewidth=3)

for s_idx, scale in enumerate(anchor_scales):
    for r_idx, ratio in enumerate(aspect_ratios):
        w = scale * np.sqrt(ratio)
        h = scale / np.sqrt(ratio)
        rect = patches.Rectangle(
            (center_x - w/2, center_y - h/2), w, h,
            linewidth=2, edgecolor=scale_colors[s_idx], facecolor='none',
            linestyle=['-', '--', ':'][r_idx]
        )
        axes[0].add_patch(rect)

# Legend for Panel 1
legend_elements = [
    Line2D([0], [0], color=scale_colors[0], lw=2, label=f'Scale {anchor_scales[0]}'),
    Line2D([0], [0], color=scale_colors[1], lw=2, label=f'Scale {anchor_scales[1]}'),
    Line2D([0], [0], color=scale_colors[2], lw=2, label=f'Scale {anchor_scales[2]}'),
    Line2D([0], [0], color='gray', lw=2, ls='-', label='Ratio 1:2 (tall)'),
    Line2D([0], [0], color='gray', lw=2, ls='--', label='Ratio 1:1 (square)'),
    Line2D([0], [0], color='gray', lw=2, ls=':', label='Ratio 2:1 (wide)'),
]
axes[0].legend(handles=legend_elements, loc='lower left', fontsize=8)
axes[0].set_title('9 Anchors at a Single Point\n(3 scales x 3 aspect ratios)', fontsize=13, fontweight='bold')
axes[0].set_xlim(0, 448)
axes[0].set_ylim(448, 0)
axes[0].axis('off')

# Panel 2: Show square anchors tiled at multiple grid points
axes[1].imshow(img_bg)
stride = 64  # Feature map stride
grid_points_x = np.arange(stride//2, 448, stride)
grid_points_y = np.arange(stride//2, 448, stride)

for gx in grid_points_x:
    for gy in grid_points_y:
        axes[1].plot(gx, gy, 'r.', markersize=4)
        # Draw only the medium-scale square anchor at each point
        sq_scale = 80
        rect = patches.Rectangle(
            (gx - sq_scale/2, gy - sq_scale/2), sq_scale, sq_scale,
            linewidth=0.8, edgecolor='cyan', facecolor='none', alpha=0.6
        )
        axes[1].add_patch(rect)

axes[1].set_title(f'Square Anchors Tiled Across Image\n({len(grid_points_x)}x{len(grid_points_y)} = '
                   f'{len(grid_points_x)*len(grid_points_y)} locations)',
                   fontsize=13, fontweight='bold')
axes[1].set_xlim(0, 448)
axes[1].set_ylim(448, 0)
axes[1].axis('off')

# Panel 3: Show ALL anchors (all 9 per location) -- densely tiled
axes[2].imshow(img_bg, alpha=0.4)
total_anchors = 0
for gx in grid_points_x:
    for gy in grid_points_y:
        for s_idx, scale in enumerate([48, 80, 120]):  # Smaller scales for visibility
            for ratio in aspect_ratios:
                w = scale * np.sqrt(ratio)
                h = scale / np.sqrt(ratio)
                rect = patches.Rectangle(
                    (gx - w/2, gy - h/2), w, h,
                    linewidth=0.3, edgecolor=scale_colors[s_idx],
                    facecolor='none', alpha=0.35
                )
                axes[2].add_patch(rect)
                total_anchors += 1

axes[2].set_title(f'All {total_anchors} Anchors Across the Image\n(Dense coverage at all scales)',
                   fontsize=13, fontweight='bold')
axes[2].set_xlim(0, 448)
axes[2].set_ylim(448, 0)
axes[2].axis('off')

plt.suptitle('Anchor Boxes in Faster R-CNN / SSD', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print(f"\nTotal anchors generated in this example: {total_anchors}")
print("Faster R-CNN generates ~20,000 anchors per image. NMS and scoring reduce these to the final detections.")

---
## Section 9: Comparing Detection Approaches

The history of object detection shows a clear trend from **slow multi-stage** pipelines to **fast single-stage** detectors:

| Method | Year | Speed (FPS) | mAP (VOC/COCO) | Approach |
|--------|------|-------------|-----------------|----------|
| **R-CNN** | 2014 | ~0.05 | 58.5 | Selective Search + CNN per region |
| **Fast R-CNN** | 2015 | ~2 | 70.0 | Single CNN + RoI pooling |
| **Faster R-CNN** | 2015 | ~7 | 73.2 | Region Proposal Network + RoI pooling |
| **SSD300** | 2016 | ~59 | 74.3 | Single shot, multi-scale feature maps |
| **YOLOv1** | 2016 | ~45 | 63.4 | Single shot, grid-based |
| **YOLOv3** | 2018 | ~30 | 57.9 (COCO) | Multi-scale, Darknet-53 backbone |

### Key Takeaways:
- **R-CNN to Faster R-CNN**: Moved from external region proposals to learned proposals (RPN), improving speed from 0.05 FPS to 7 FPS
- **YOLO/SSD**: Eliminated the region proposal step entirely -- single forward pass predicts all boxes and classes simultaneously
- **The trade-off**: Single-stage detectors are much faster but historically slightly less accurate on small objects. Modern versions (YOLOv5, YOLOv8) have largely closed this gap.

In [ ]:
# Visualize the speed vs accuracy trade-off
from matplotlib.patches import Patch as MPatch

methods = ['R-CNN', 'Fast R-CNN', 'Faster R-CNN', 'SSD300', 'YOLOv1', 'YOLOv3']
fps_values = [0.05, 2, 7, 59, 45, 30]
map_values = [58.5, 70.0, 73.2, 74.3, 63.4, 57.9]
years = [2014, 2015, 2015, 2016, 2016, 2018]
types = ['Multi-stage', 'Multi-stage', 'Multi-stage', 'Single-stage', 'Single-stage', 'Single-stage']

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left plot: Speed vs Accuracy scatter
scatter_colors = ['#F44336' if t == 'Multi-stage' else '#2196F3' for t in types]

for i, (method, fps, mAP, color) in enumerate(zip(methods, fps_values, map_values, scatter_colors)):
    axes[0].scatter(fps, mAP, c=color, s=200, zorder=5, edgecolors='black', linewidths=1.5)
    offset_x = 1.5 if fps < 50 else -8
    offset_y = 1.2
    axes[0].annotate(method, (fps, mAP), textcoords="offset points",
                     xytext=(offset_x, offset_y), fontsize=11, fontweight='bold')

axes[0].set_xlabel('Speed (FPS)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('mAP (%)', fontsize=13, fontweight='bold')
axes[0].set_title('Speed vs. Accuracy Trade-off', fontsize=14, fontweight='bold')
axes[0].set_xscale('log')
axes[0].grid(True, alpha=0.3)

# Custom legend
legend_elems = [
    MPatch(facecolor='#F44336', edgecolor='black', label='Multi-stage (R-CNN family)'),
    MPatch(facecolor='#2196F3', edgecolor='black', label='Single-stage (YOLO/SSD)'),
]
axes[0].legend(handles=legend_elems, loc='lower right', fontsize=11)

# Right plot: Speed bar chart
bar_colors = ['#EF5350', '#EF5350', '#EF5350', '#42A5F5', '#42A5F5', '#42A5F5']
bars = axes[1].barh(methods, fps_values, color=bar_colors, edgecolor='black', linewidth=0.8)
axes[1].set_xlabel('Speed (FPS -- Frames Per Second)', fontsize=13, fontweight='bold')
axes[1].set_title('Detection Speed Comparison', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()

# Add value labels on bars
for bar, fps in zip(bars, fps_values):
    axes[1].text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{fps} FPS', va='center', fontsize=11, fontweight='bold')

# Add real-time threshold line
axes[1].axvline(x=30, color='green', linestyle='--', linewidth=2, alpha=0.7)
axes[1].text(31, 0.2, 'Real-time\n(30 FPS)', color='green', fontsize=10, fontweight='bold')

axes[1].set_xlim(0, 70)
axes[1].grid(True, axis='x', alpha=0.3)

plt.suptitle('Evolution of Object Detection: From R-CNN to YOLO',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("The trend is clear: from multi-stage (R-CNN) to single-stage (YOLO/SSD),")
print("trading a small amount of accuracy for dramatically faster inference.")
print("\nModern detectors like YOLOv5/v8 achieve both high speed AND high accuracy.")

---
## Summary

In this notebook, we covered the **core concepts of object detection**:

1. **Classification vs. Detection** -- Detection finds multiple objects and their locations
2. **IoU (Intersection over Union)** -- The fundamental metric for evaluating detection quality (threshold: 0.5)
3. **NMS (Non-Maximum Suppression)** -- Removes redundant overlapping detections
4. **YOLO's Grid Approach** -- Divides image into S x S cells, each predicting B boxes and C class probabilities
5. **Pretrained Detection** -- Using Faster R-CNN (torchvision) for real object detection
6. **Multi-Object Detection** -- Detecting and localizing multiple objects simultaneously
7. **Anchor Boxes** -- Fixed reference boxes that the network refines with predicted offsets
8. **Detection Methods Comparison** -- From slow R-CNN (0.05 FPS) to real-time YOLO/SSD (45+ FPS)

### For Further Reading:
- Redmon et al., "You Only Look Once: Unified, Real-Time Object Detection" (CVPR 2016)
- Ren et al., "Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks" (NeurIPS 2015)
- Liu et al., "SSD: Single Shot MultiBox Detector" (ECCV 2016)

---
*ELC 5365 -- Deep Learning, Baylor University*